# Задание 1. Обучение нейросети, EDA и продвинутая предобработка (MICE + SMOTE)

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.neural_network import MLPClassifier
from sklearn.utils import resample
import joblib

import warnings
warnings.filterwarnings('ignore')

## 1. Загрузка и анализ корреляций

In [2]:
df = pd.read_csv("weather.csv")
print(f"Изначальный размер датасета: {df.shape}")

# Удаление дублей с высокой корреляцией (> 0.85)
numeric_df = df.select_dtypes(include=['int64', 'float64'])
corr_matrix = numeric_df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.85)]
print(f"Удаляем избыточные признаки-дубли: {to_drop}")

df = df.drop(columns=to_drop)

# Удаляем Sunshine и Evaporation (слишком много пропусков)
cols_to_remove = [c for c in ['Sunshine', 'Evaporation'] if c in df.columns]
df = df.drop(columns=cols_to_remove)

# Удаляем строки, где нет целевой переменной
df = df.dropna(subset=['RainTomorrow'])

Изначальный размер датасета: (145460, 23)
Удаляем избыточные признаки-дубли: ['Pressure3pm', 'Temp9am', 'Temp3pm']


## 2. Oversampling (Борьба с дисбалансом классов)
Чтобы модель лучше предсказывала дождь (класс 1), мы сбалансируем классы, скопировав часть данных меньшинства. Это резко повышает F1-Score.

In [ ]:
majority = df[df['RainTomorrow'] == 'No']
minority = df[df['RainTomorrow'] == 'Yes']

# Увеличиваем класс меньшинства до размера мажоритарного
minority_upsampled = resample(minority, 
                              replace=True,     
                              n_samples=len(majority),    
                              random_state=42)

oversampled = pd.concat([majority, minority_upsampled])
print(f"Размер датасета после балансировки: {oversampled.shape}")
print("Распределение таргета:\n", oversampled['RainTomorrow'].value_counts())

Размер датасета после балансировки: (220632, 18)
Распределение таргета:
 RainTomorrow
No     110316
Yes    110316
Name: count, dtype: int64


## 3. Продвинутая предобработка (Label Encoding + MICE)
Заменяем пропуски в категориальных фичах на моду, применяем LabelEncoder, а затем используем мощный IterativeImputer (MICE) для числовых фич.

In [4]:
# Выделяем месяц из даты и удаляем саму дату
oversampled['Date'] = pd.to_datetime(oversampled['Date'])
oversampled['Month'] = oversampled['Date'].dt.month
oversampled = oversampled.drop(columns=['Date'])

# 1. Impute categorical vars with Mode
categorical_cols = oversampled.select_dtypes(include=['object', 'string']).columns
for col in categorical_cols:
    oversampled[col] = oversampled[col].fillna(oversampled[col].mode()[0])

# 2. Label Encoding
lencoders = {}
for col in categorical_cols:
    lencoders[col] = LabelEncoder()
    oversampled[col] = lencoders[col].fit_transform(oversampled[col])

print("Все категориальные признаки успешно закодированы.")

Все категориальные признаки успешно закодированы.


In [5]:
# 3. Multiple Imputation by Chained Equations (MICE)
print("Запуск MICE для восстановления числовых пропусков (может занять 1-2 минуты)...")
mice_imputer = IterativeImputer(max_iter=10, random_state=42)

# Применяем MICE ко всему датасету (MiceImputed)
MiceImputed = pd.DataFrame(mice_imputer.fit_transform(oversampled), columns=oversampled.columns)
print("Восстановление пропусков завершено!")

Запуск MICE для восстановления числовых пропусков (может занять 1-2 минуты)...


KeyboardInterrupt: 

## 4. Разделение и масштабирование данных

In [ ]:
X = MiceImputed.drop(columns=['RainTomorrow'])
y = MiceImputed['RainTomorrow']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Масштабируем данные для нейросети (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

NameError: name 'MiceImputed' is not defined

## 5. Обучение модели и поиск гиперпараметров
Используем структуру из лучшего решения Kaggle: (30, 30, 30) слои, активация logistic и solver lbfgs/adam.

In [ ]:
model_mlp = MLPClassifier(max_iter=500, random_state=42, verbose=True)

mlp_param_grid = {
    'hidden_layer_sizes': [(100, 3), (3, 100), (10, 30), (30, 10), (10, 20, 10), (100, 30, 10), (30, 30, 30)],
    'activation': ['logistic', 'relu'],
    'solver': ['adam'],
    'alpha': [0.0001, 0.05]
}

print("Начинаем GridSearchCV...")
grid_search = GridSearchCV(
    model_mlp, 
    param_grid=mlp_param_grid, 
    cv=3, 
    scoring='f1',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train_scaled, y_train)

print("Поиск завершен!")
print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучший F1 скор: {grid_search.best_score_}")

## 6. Сохранение лучшей модели и результатов перебора параметров

In [ ]:
# 1. Сохраняем лучшую модель, скейлер и тестовые данные
best_model = grid_search.best_estimator_
joblib.dump(best_model, 'best_mlp.pkl')
joblib.dump(scaler, 'scaler.pkl') # Сохраняем scaler для inference!
joblib.dump((X_test_scaled, y_test), 'test_data.pkl')
print("Лучшая модель, скейлер и тестовые данные сохранены.")

# 2. Сохраняем результаты всех моделей (какие параметры какой скор дали)
results_df = pd.DataFrame(grid_search.cv_results_)
cols_to_keep = ['params', 'mean_test_score', 'std_test_score', 'rank_test_score', 'mean_fit_time']
results_df = results_df[cols_to_keep].sort_values(by='rank_test_score')

# Сохраняем в CSV, чтобы можно было удобно посмотреть результаты
results_df.to_csv('grid_search_results.csv', index=False)
print("\nРезультаты для всех комбинаций гиперпараметров сохранены в 'grid_search_results.csv'")

display(results_df.head())